# Import Required Libraries

In [88]:
import os
import re
import warnings
warnings.filterwarnings("ignore")
from tqdm import tqdm 

# Decorter packages
from typing import List

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Vis for EEG Analysis
import mne

from sklearn.model_selection import train_test_split

In [3]:
try:
    assert(len([i for i in os.listdir("../data/") if i.startswith("S")]) == 60)
    file_dir = [i for i in os.listdir("../data/") if i.startswith("S")]
    print("All subjects directories are loaded!!")
except:
    print("Error!! Not all subjects directories are loaded")

All subjects directories are loaded!!


In [7]:
print("Total Number of Subjects (Analyzed): ", len(file_dir))

Total Number of Subjects (Analyzed):  60


# 1. Get MetaData File Structure

In [40]:
def metadata_info(file_folder: str, focusGroup = "M", baselocation = "../data/"):
    """
        Function: This function contains the meta data information about the files
        args:
            file_dir: list of file folders 
    """
    file_folder = os.path.join(baselocation, file_folder)  
    file_folder_dir_lst = os.listdir(file_folder)

    # Retrieve files that contains Motor/Imagery related EEG signals
    focused_file_lst = [file for file in file_folder_dir_lst if focusGroup in file]
    focused_file_lst = [file for file in focused_file_lst if focusGroup+"8" not in file]
    focused_file_lst = [file for file in focused_file_lst if focusGroup+"1" not in file] 

    # Extract the required identifiers Subject ID, 
    #                                  Repetition Number, 
    #                                  Motor or Motor Imagery Activity, 
    #                                  label 
    #                                  and Task Repetition Number

    def re_match(filename:str):
        """ Matches the filename into the above mentioned respective groups"""
        match = re.match(rf'(S\d+)(R\d+)({focusGroup}\d+)_([0-9]+)\.csv', filename)

        s_part = match.group(1) # Subject ID component
        r_part = match.group(2) # Repetition ID component
        m_part = match.group(3) # Motor Tasks
        number = int(match.group(4)) # Repetition

        return (filename, s_part, r_part, m_part, number)

    # Create a Pandas dataframe that holds the metadata information and sort based on motor label and Task Repetition No
    meta_info =[re_match(file) for file in focused_file_lst]
    meta_info = pd.DataFrame(meta_info, columns = ["FileName", "SubjectID", "RepetitionNo", "Motor_label", "Task_RepetitionNo"]) 
    meta_info = meta_info.sort_values(by=["Motor_label", "Task_RepetitionNo"], ascending=True).reset_index(drop= True) 

    return meta_info

consolidated_metadata_info = [metadata_info(file_folder=file_str, focusGroup = "I") for file_str in file_dir]
consolidated_metadata_info = pd.concat(consolidated_metadata_info).reset_index(drop = True)

In [ ]:
consolidated_metadata_info[consolidated_metadata_info["SubjectID"] == "S34"].\
                                sort_values(ascending = True,
                                by = ["SubjectID", "RepetitionNo", "Task_RepetitionNo"])

,FileName,SubjectID,RepetitionNo,Motor_label,Task_RepetitionNo
0,S34R1I2_1.csv,S34,R1,I2,1
5,S34R1I3_1.csv,S34,R1,I3,1
10,S34R1I4_1.csv,S34,R1,I4,1
15,S34R1I5_1.csv,S34,R1,I5,1
20,S34R1I6_1.csv,S34,R1,I6,1
25,S34R1I7_1.csv,S34,R1,I7,1
1,S34R1I2_2.csv,S34,R1,I2,2
6,S34R1I3_2.csv,S34,R1,I3,2
11,S34R1I4_2.csv,S34,R1,I4,2
16,S34R1I5_2.csv,S34,R1,I5,2


# 2. Generate EEG Data - Trail Based 3D Array

-  Where X_trails.shape -> (n_trails, n_channels, n_samples)
-  Where y_trails.shape -> (n_trails, )

In [89]:
def GenerateNumpyArray(consolidated_metadata_info: pd.DataFrame, 
                       subject_base_location = "../data/"):

    X, y = [], []
    for file_name, subID, label in tqdm(zip(consolidated_metadata_info["FileName"],
                                       consolidated_metadata_info["SubjectID"],
                                       consolidated_metadata_info["Motor_label"])):
        
        file_path = os.path.join(subject_base_location, subID, file_name)

        # EEG Signals - Input 
        data_eeg = (pd.read_csv(file_path)
                    .iloc[:, 1:]
                    .to_numpy()).T # (n_channels, n_samples)
        
        X.append(data_eeg)


        # EEG Signals - label
        y.append(int(label[1:]))

    X = np.array(X)
    y = np.array(y)

    return X, y


X, y = GenerateNumpyArray(consolidated_metadata_info)
print(f"EEG Signals X - (n_trails, n_channels, n_samples) : {X.shape}")
print(f"EEG Signals y - (n_trails, ) : {y.shape}" )

1799it [00:02, 828.54it/s]

EEG Signals X - (n_trails, n_channels, n_samples) : (1799, 16, 500)
EEG Signals y - (n_trails, ) : (1799,)


# 3. Save Data

In [93]:
np.savez_compressed("../eeg_processed/eeg_data.npz", 
                    X = X, 
                    y = y)

# Check
data = np.load("../eeg_processed/eeg_data.npz")
data["X"].shape, data["y"].shape

((1799, 16, 500), (1799,))